# Cache-and-Persist-in-Spark

In PySpark, **`cache()`** and **`persist()`** are optimization techniques used to store (or *persist*) intermediate computation results in memory or on disk, avoiding redundant calculations and speeding up data processing. Here's a detailed comparison: 

In [40]:
from pyspark.sql import SparkSession
from pyspark.storagelevel import *
spark = SparkSession.builder.appName("cacheAndPersist").master("local[5]").getOrCreate()
spark

In [41]:
userDF = spark.read.format("parquet")\
        .option("inferSchema", "true")\
        .option("header","true")\
        .load("data/userdata.parquet")
userDF = userDF.repartition(4)

# cache()
- **Purpose**: Store data in memory for fast reuse.  
- **Storage Level**: Uses the default storage level **`MEMORY_ONLY`**.  
  - Data is stored as **deserialized Java objects** in the JVM heap.  
  - If memory is insufficient, partitions that don't fit are **recomputed on-the-fly** (not cached). 

In [42]:
userDF.cache()

DataFrame[registration_dttm: timestamp, id: int, first_name: string, last_name: string, email: string, gender: string, ip_address: string, cc: string, country: string, birthdate: string, salary: double, title: string, comments: string]

In [43]:
userDF.show() # only one partition will come in storage pull location

+-------------------+---+----------+---------+--------------------+------+---------------+-------------------+------------+----------+---------+--------------------+-----------+
|  registration_dttm| id|first_name|last_name|               email|gender|     ip_address|                 cc|     country| birthdate|   salary|               title|   comments|
+-------------------+---+----------+---------+--------------------+------+---------------+-------------------+------------+----------+---------+--------------------+-----------+
|2016-02-03 09:50:49|705|          |    Riley|                    |Female|  39.227.25.235|                   |   Indonesia| 11/1/1977|115133.62|Senior Financial ...|åß∂ƒ©˙∆˚¬…æ|
|2016-02-03 06:09:29|655|    Johnny|     Reed|jreedi6@chicagotr...|  Male|169.161.103.111|   4844445630272291|      Russia| 5/23/1979| 68913.72|    Quality Engineer|           |
|2016-02-03 07:17:51|534|   Lillian|   Pierce|  lpierceet@narod.ru|Female|186.129.141.195|   3544968114106706|

**💡 only one partition will come in storage pull location when show method will call**

In [44]:
userDF.count() # all partition will come in storage pull location

1000

In [45]:
userDF.unpersist() # Free up memory/disk

DataFrame[registration_dttm: timestamp, id: int, first_name: string, last_name: string, email: string, gender: string, ip_address: string, cc: string, country: string, birthdate: string, salary: double, title: string, comments: string]

# Persist()
- **Purpose**: Store data with **flexible storage levels** (memory, disk, or both).  
- **Storage Levels** (via `pyspark.StorageLevel`):  

| Level                                           | Description                           | Use Case                         |
| ----------------------------------------------- | ------------------------------------- | -------------------------------- |
| `MEMORY_ONLY`                                   | Default (same as `cache()`)           | Fast access when memory suffices |
| `MEMORY_AND_DISK`                               | Spill to disk if memory full          | Large datasets                   |
| `MEMORY_ONLY_SER`                               | Serialized objects (saves space)      | Memory-constrained clusters      |
| `MEMORY_AND_DISK_SER`                           | Serialized + disk spill               | Very large datasets              |
| `DISK_ONLY`                                     | Store only on disk                    | Cheapest option, slowest access  |
| `OFF_HEAP`                                      | Store outside JVM heap (experimental) | Avoid garbage collection pauses  |
| Levels with `_2` suffix (e.g., `MEMORY_ONLY_2`) | Replicate partitions on 2 nodes       | Fault tolerance                  |


In [46]:
persistDF = userDF.filter(userDF.country=="China").filter(userDF.salary>100000)
persistDF.persist(StorageLevel.MEMORY_AND_DISK)
persistDF.show()


+-------------------+---+----------+----------+--------------------+------+---------------+-------------------+-------+----------+---------+--------------------+----------------------+
|  registration_dttm| id|first_name| last_name|               email|gender|     ip_address|                 cc|country| birthdate|   salary|               title|              comments|
+-------------------+---+----------+----------+--------------------+------+---------------+-------------------+-------+----------+---------+--------------------+----------------------+
|2016-02-04 03:59:46|221|     Robin|Montgomery|rmontgomery64@uol...|Female| 117.255.30.224|   4508764341809721|  China|  1/2/1996|111349.02|Human Resources A...|         `⁄€‹›ﬁﬂ‡°·‚—±|
|2016-02-03 14:33:28|845|  Kimberly|    Burton|kburtonng@geociti...|Female| 26.240.110.184|   3544797709421000|  China| 7/19/1977|281001.88|Staff Accountant III|                  NULL|
|2016-02-03 05:46:47|314|     James|    Harvey|   jharvey8p@npr.org|  Male|

In [47]:
persistDF.count()

128

In [49]:
persistDF.unpersist() # Free up memory/disk

DataFrame[registration_dttm: timestamp, id: int, first_name: string, last_name: string, email: string, gender: string, ip_address: string, cc: string, country: string, birthdate: string, salary: double, title: string, comments: string]

#### **💡 Best practice: Always free up memory/disk resources by unpersisting DataFrames when they're no longer needed**